In [1]:
import pandas as pd
import numpy as np
import json

# Constants matching the JavaScript configuration
NUM_FEATURES = 8
CAT_FEATURES = 2
TOTAL = NUM_FEATURES + CAT_FEATURES
CAT_CLASSES = [3, 2]
SAMPLE_SIZE = 200

# Sample frequency data for categorical features
CAT_FREQS = [
    [120, 80, 50],
    [150, 100]
]

# Distribution types for numeric features
DISTRIBUTION_TYPES = [
    'uniform', 'exponential', 'bimodal', 'lognormal', 'beta', 'gamma', 'chiSquared', 'weibull'
]

def next_gaussian():
    u, v = 0, 0
    while u == 0:
        u = np.random.random()
    while v == 0:
        v = np.random.random()
    return np.sqrt(-2.0 * np.log(u)) * np.cos(2.0 * np.pi * v)

def beta_dist(alpha, beta):
    u1 = np.power(np.random.random(), 1/alpha)
    u2 = np.power(np.random.random(), 1/beta)
    return u1 / (u1 + u2)

def gamma_dist(k, theta):
    if k > 1:
        d = k - 1/3
        c = 1 / np.sqrt(9 * d)
        
        while True:
            while True:
                z = next_gaussian()
                v = np.power(1 + c * z, 3)
                if v > 0:
                    break
            
            u = np.random.random()
            if u <= 1 - 0.331 * np.power(z, 4) and np.log(u) <= 0.5 * z*z + d*(1 - v + np.log(v)):
                return d * v * theta
    else:
        return gamma_dist(k + 1, theta) * np.power(np.random.random(), 1/k)

def weibull_dist(lambda_, k):
    return lambda_ * np.power(-np.log(1 - np.random.random()), 1/k)

def generate_sample_data():
    feature_data = {}
    
    for feature_index in range(NUM_FEATURES):
        dist_type = DISTRIBUTION_TYPES[feature_index % len(DISTRIBUTION_TYPES)]
        data = []
        
        if dist_type == 'uniform':
            data = np.random.uniform(0, 100, SAMPLE_SIZE).tolist()
            
        elif dist_type == 'exponential':
            data = [-np.log(1 - np.random.random()) * 20 for _ in range(SAMPLE_SIZE)]
            
        elif dist_type == 'bimodal':
            for i in range(SAMPLE_SIZE):
                if np.random.random() < 0.6:
                    data.append(30 + (next_gaussian() * 10))
                else:
                    data.append(70 + (next_gaussian() * 8))
                    
        elif dist_type == 'lognormal':
            data = [np.exp(next_gaussian() * 0.8 + 3.5) for _ in range(SAMPLE_SIZE)]
            
        elif dist_type == 'beta':
            data = [beta_dist(2, 5) * 100 for _ in range(SAMPLE_SIZE)]
            
        elif dist_type == 'gamma':
            data = [gamma_dist(2, 2) * 10 for _ in range(SAMPLE_SIZE)]
            
        elif dist_type == 'chiSquared':
            data = []
            for i in range(SAMPLE_SIZE):
                sum_val = 0
                for j in range(4):
                    normal = next_gaussian()
                    sum_val += normal * normal
                data.append(sum_val * 5)
                
        elif dist_type == 'weibull':
            data = [weibull_dist(1, 1.5) * 30 for _ in range(SAMPLE_SIZE)]
        
        # Calculate statistics
        mean_val = np.mean(data)
        std_val = np.std(data)
        
        feature_data[feature_index] = {
            'values': data,
            'type': dist_type,
            'mean': mean_val,
            'std': std_val
        }
    
    return feature_data

def save_data_to_files(feature_data):
    # Save numeric features data
    numeric_data = {}
    for i in range(NUM_FEATURES):
        numeric_data[f'feature_{i}'] = feature_data[i]['values']
        numeric_data[f'feature_{i}_type'] = [feature_data[i]['type']] * SAMPLE_SIZE
        numeric_data[f'feature_{i}_mean'] = [feature_data[i]['mean']] * SAMPLE_SIZE
        numeric_data[f'feature_{i}_std'] = [feature_data[i]['std']] * SAMPLE_SIZE
    
    df_numeric = pd.DataFrame(numeric_data)
    df_numeric.to_csv('numeric_features.csv', index=False, encoding='utf-8')
    
    # Save categorical features data
    categorical_data = {
        'cat_feature_0': [0] * CAT_FREQS[0][0] + [1] * CAT_FREQS[0][1] + [2] * CAT_FREQS[0][2],
        'cat_feature_1': [0] * CAT_FREQS[1][0] + [1] * CAT_FREQS[1][1]
    }
    df_categorical = pd.DataFrame(categorical_data)
    df_categorical.to_csv('categorical_features.csv', index=False, encoding='utf-8')
    
    # Save metadata
    metadata = {
        'NUM_FEATURES': NUM_FEATURES,
        'CAT_FEATURES': CAT_FEATURES,
        'TOTAL': TOTAL,
        'CAT_CLASSES': CAT_CLASSES,
        'SAMPLE_SIZE': SAMPLE_SIZE,
        'CAT_FREQS': CAT_FREQS,
        'DISTRIBUTION_TYPES': DISTRIBUTION_TYPES,
        'feature_stats': {}
    }
    
    for i in range(NUM_FEATURES):
        metadata['feature_stats'][f'feature_{i}'] = {
            'type': feature_data[i]['type'],
            'mean': feature_data[i]['mean'],
            'std': feature_data[i]['std']
        }
    
    with open('metadata.json', 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    
    print("Data saved successfully!")
    print(f"- numeric_features.csv: {NUM_FEATURES} numeric features")
    print(f"- categorical_features.csv: {CAT_FEATURES} categorical features") 
    print(f"- metadata.json: configuration and statistics")
    
    return df_numeric, df_categorical

if __name__ == "__main__":
    # Generate sample data
    feature_data = generate_sample_data()
    
    # Save to files
    df_numeric, df_categorical = save_data_to_files(feature_data)
    
    print("\nData generation completed successfully!")
    for i in range(NUM_FEATURES):
        print(f"Feature {i}: {feature_data[i]['type']} - Mean: {feature_data[i]['mean']:.2f}, Std: {feature_data[i]['std']:.2f}")

Data saved successfully!
- numeric_features.csv: 8 numeric features
- categorical_features.csv: 2 categorical features
- metadata.json: configuration and statistics

Data generation completed successfully!
Feature 0: uniform - Mean: 49.86, Std: 28.85
Feature 1: exponential - Mean: 18.76, Std: 19.25
Feature 2: bimodal - Mean: 45.33, Std: 22.44
Feature 3: lognormal - Mean: 52.12, Std: 51.84
Feature 4: beta - Mean: 42.90, Std: 12.10
Feature 5: gamma - Mean: 33.75, Std: 14.38
Feature 6: chiSquared - Mean: 18.92, Std: 13.83
Feature 7: weibull - Mean: 29.25, Std: 19.95
